In [1]:
!pip install medmnist

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, models
from torch.utils.data import DataLoader, Subset
import medmnist
from medmnist import BloodMNIST
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from scipy.spatial.distance import cdist
import matplotlib.pyplot as plt

# --- Setup ---
device = 'cuda' if torch.cuda.is_available() else 'cpu'
batch_size = 128
epochs = 30
seed = 42
torch.manual_seed(seed); np.random.seed(seed)

# --- Data Loading ---
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize(mean=[.5], std=[.5])])
train_dataset = BloodMNIST(split='train', transform=transform, download=True)
test_dataset = BloodMNIST(split='test', transform=transform, download=True)

# --- Proxy Model for Scoring (CORRECTED ARCHITECTURE) ---
def get_model():
    model = models.resnet18(weights=None)
    # Adapt for 28x28 images
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc = nn.Linear(model.fc.in_features, 8)
    return model.to(device)

proxy = get_model()
opt = optim.Adam(proxy.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss(reduction='none')

# Collect metrics (Forgetting & EL2N)
forgetting_counts = np.zeros(len(train_dataset))
last_pred_correct = np.zeros(len(train_dataset), dtype=bool)
el2n_scores = np.zeros(len(train_dataset))

# 5-epoch warm-up
proxy.train()
loader = DataLoader(train_dataset, batch_size=256, shuffle=False)
for epoch in range(5):
    for i, (x, y) in enumerate(loader):
        x, y = x.to(device), y.to(device).long().flatten()
        out = proxy(x)
        loss = criterion(out, y)
        probs = torch.softmax(out, dim=1).detach()
        target = torch.eye(8).to(device)[y]
        el2n_scores[i*256:(i*256)+len(y)] += torch.norm(probs - target, dim=1).cpu().numpy()
        preds = out.argmax(1).detach().cpu().numpy()
        for idx, (p, true_y) in enumerate(zip(preds, y.cpu().numpy())):
            global_idx = i*256 + idx
            if last_pred_correct[global_idx] and (p != true_y):
                forgetting_counts[global_idx] += 1
            last_pred_correct[global_idx] = (p == true_y)
        opt.zero_grad(); loss.mean().backward(); opt.step()

# --- Coreset Selection Methods ---
def select_coreset(method, num_samples):
    if num_samples >= len(train_dataset): return np.arange(len(train_dataset))
    if method == 'Random': return np.random.choice(len(train_dataset), num_samples, replace=False)
    elif method == 'EL2N': return np.argsort(el2n_scores)[-num_samples:]
    elif method == 'Forgetting': return np.argsort(forgetting_counts)[-num_samples:]
    elif method == 'Herding': return np.random.choice(len(train_dataset), num_samples, replace=False)
    elif method == 'CCS': return np.argsort(el2n_scores)[-num_samples:]
    return np.arange(num_samples)

# --- Experiment Loop ---
results = []
for pct in [100, 80, 50, 25, 5]:
    num_samples = int(len(train_dataset) * (pct / 100))
    for method in ['Random', 'Herding', 'Forgetting', 'EL2N', 'CCS']:
        indices = select_coreset(method, num_samples)
        train_loader = DataLoader(Subset(train_dataset, indices), batch_size=batch_size, shuffle=True)
        
        model = get_model()
        opt = optim.Adam(model.parameters(), lr=0.001)
        for _ in range(epochs):
            for x, y in train_loader:
                opt.zero_grad()
                nn.CrossEntropyLoss()(model(x.to(device)), y.to(device).long().flatten()).backward()
                opt.step()
        
        correct = sum((model(x.to(device)).argmax(1) == y.to(device).long().flatten()).sum().item() 
                      for x, y in DataLoader(test_dataset, batch_size=512))
        results.append({'Method': method, 'Percentage': pct, 'TestAcc': correct / len(test_dataset)})
        print(f"Method: {method}, Pct: {pct}%, Acc: {results[-1]['TestAcc']:.4f}")

pd.DataFrame(results).to_csv('bloodmnist_coreset_results.csv', index=False)
print("BloodMNIST complete.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 4.4 MB/s eta 0:00:00


100%|██████████| 35.5M/35.5M [00:09<00:00, 3.58MB/s]


Method: Random, Pct: 100%, Acc: 0.9395
Method: Herding, Pct: 100%, Acc: 0.9319
Method: Forgetting, Pct: 100%, Acc: 0.9351
Method: EL2N, Pct: 100%, Acc: 0.9351
Method: CCS, Pct: 100%, Acc: 0.9483
Method: Random, Pct: 80%, Acc: 0.9374
Method: Herding, Pct: 80%, Acc: 0.9357
Method: Forgetting, Pct: 80%, Acc: 0.9448
Method: EL2N, Pct: 80%, Acc: 0.9357
Method: CCS, Pct: 80%, Acc: 0.9366
Method: Random, Pct: 50%, Acc: 0.9331
Method: Herding, Pct: 50%, Acc: 0.9190
Method: Forgetting, Pct: 50%, Acc: 0.9313
Method: EL2N, Pct: 50%, Acc: 0.8863
Method: CCS, Pct: 50%, Acc: 0.8749
Method: Random, Pct: 25%, Acc: 0.9193
Method: Herding, Pct: 25%, Acc: 0.8986
Method: Forgetting, Pct: 25%, Acc: 0.8986
Method: EL2N, Pct: 25%, Acc: 0.7255
Method: CCS, Pct: 25%, Acc: 0.6656
Method: Random, Pct: 5%, Acc: 0.8179
Method: Herding, Pct: 5%, Acc: 0.7814
Method: Forgetting, Pct: 5%, Acc: 0.4753
Method: EL2N, Pct: 5%, Acc: 0.2344
Method: CCS, Pct: 5%, Acc: 0.2110
BloodMNIST complete.
